In [84]:
import bw2analyzer as ba
import bw2data as bd
import bw2calc as bc
import bw2io as bi
import matrix_utils as mu
import bw_processing as bp
import time
import wurst
import os
from pathlib import Path
import copy
import pandas as pd

In [85]:
start_time = time.time()

In [86]:
bd.projects.set_current('Nitrous oxide production')

In [87]:
bd.databases

Databases dictionary with 6 object(s):
	ecoinvent-3.10-biosphere
	ecoinvent-3.10-cutoff
	ecoinvent-3.10-cutoff-Base-2020
	ecoinvent-3.10-cutoff-Base-2050
	ecoinvent-3.10-cutoff-RCP19-2050
	ecoinvent-3.10-cutoff-RCP26-2050

In [88]:
# ei_db = wurst.extract_brightway2_databases('ecoinvent-3.10-cutoff-Base-2020')

In [89]:
# ei_db = bd.Database('ecoinvent-3.10-cutoff-Base-2020')

In [90]:
# for ds in ei_db:
#     if 'market for hydrogen, gaseous, medium pressure, merchant' in ds['name'] and 'kilogram' in ds['unit']:
#         print(ds['reference product'], ds['name'], ds['location'], ds['unit'])
#         act = ds

In [91]:
# lca = bc.LCA({act: 1}, method=("IPCC 2021", "climate change", "GWP 100a, incl. H and bio CO2"))
# lca.lci()
# lca.lcia()
# lca.score * 0.0031

In [92]:
# import inventories from excel
excel_import = bi.ExcelImporter(os.path.join('..', 'data', 'inventories_ei310.xlsx'))
# excel_import = bi.ExcelImporter(os.path.join('..', 'data', 'inventories.xlsx'))
excel_import.apply_strategies()
excel_import.match_database('ecoinvent-3.10-cutoff-Base-2020', fields = ('name', 'reference product', 'unit', 'location')) # match flows from ecoinvent database
excel_import.match_database('ecoinvent-3.10-biosphere', fields = ('name', 'unit', 'categories')) # match flows from biosphere database
excel_import.match_database(fields = ('name', 'reference product', 'unit', 'location')) # match flows from new database
excel_import.statistics()

Extracted 4 worksheets in 0.03 seconds
Applying strategy: csv_restore_tuples
Applying strategy: csv_restore_booleans
Applying strategy: csv_numerize
Applying strategy: csv_drop_unknown
Applying strategy: csv_add_missing_exchanges_section
Applying strategy: normalize_units
Applying strategy: normalize_biosphere_categories
Applying strategy: normalize_biosphere_names


Applying strategy: strip_biosphere_exc_locations
Applying strategy: set_code_by_activity_hash
Applying strategy: link_iterable_by_fields
Applying strategy: assign_only_product_as_production
Applying strategy: link_technosphere_by_activity_hash
Applying strategy: drop_falsey_uncertainty_fields_but_keep_zeros
Applying strategy: convert_uncertainty_types_to_integers
Applying strategy: convert_activity_parameters_to_list
Applied 16 strategies in 11.35 seconds
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
13 datasets
	109 exchanges
	Links to the following databases:
		ecoinvent-3.10-cutoff-Base-2020 (57 exchanges)
		nitrous-oxide-ei310-Base-2020 (29 exchanges)
		ecoinvent-3.10-biosphere (23 exchanges)
	0 unlinked exchanges (0 types)
		


(13, 109, 0, 0)

In [93]:
excel_import.write_database()

100%|██████████| 13/13 [00:00<00:00, 1184.96it/s]


18:00:16 [info     ] Vacuuming database            
Created database: nitrous-oxide-ei310-Base-2020


In [94]:
act = [ds for ds in bd.Database('nitrous-oxide-ei310-Base-2020') if 'hydrogen peroxide production, AO real, green hydrogen' == ds['name'] and 'kilogram' in ds['unit']][0]
act

'hydrogen peroxide production, AO real, green hydrogen' (kilogram, GLO, None)

In [95]:
method = ("IPCC 2021", "climate change", "GWP 100a, incl. H and bio CO2")

In [96]:
lca = bc.LCA({act: 1}, method = method)
lca.lci()
lca.lcia()
lca.score

0.48772762861720786